# RQ2 — Repeat length and sequence patterning

In [ ]:
from pathlib import Path
import os
from functools import partial
from dataclasses import replace
import sys
import pandas as pd
import matplotlib.pyplot as plt

here = Path.cwd().resolve()
ROOT = next(
    (p for p in (here, *here.parents)
     if (p / "analysis" / "src" / "plots" / "style.py").is_file()),
    None
)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter from inside the repository checkout.")
sys.path.insert(0, str(ROOT / "analysis"))

from src.plots.style import PlotStyle
from src.plots.data import load_sweep, filter_rows
from src.plots.panels import panels_by

STYLE = PlotStyle()
plt.rcParams.update(STYLE.mpl_rc())
OUTPUT_ROOT = Path(os.environ.get("RNA_ANALYSIS_RESULTS_ROOT", str(ROOT / "analysis_results"))).expanduser()

SAVE_FIGURES = True
FIGURE_ROOT = OUTPUT_ROOT / "51a_repeat_number" / "figures"
if SAVE_FIGURES:
    FIGURE_ROOT.mkdir(parents=True, exist_ok=True)

def save_figure(fig, filename: str) -> None:
    """Save a notebook figure alongside its sweep's numerical results."""
    if SAVE_FIGURES:
        fig.savefig(FIGURE_ROOT / filename, dpi=250, bbox_inches="tight")


In [ ]:
from src.plots.data import independent_units
from src.plots.figures import plot_cluster_bins, plot_ranked_clusters, plot_metric_points

PERIODIC_SWEEP = "51a_repeat_number"
SCRAMBLED_SWEEP = "51b_repeat_number_rand"
DENSITY = 0.01
STIFFNESS = 2.5
E_VALUES = (2, 4, 8)
REPEAT_COUNTS = (10, 20, 31, 47)

periodic = filter_rows(
    load_sweep(OUTPUT_ROOT, PERIODIC_SWEEP, "summary"),
    density=DENSITY, angle_k_ppp=STIFFNESS, E_att=E_VALUES, repeat_count=REPEAT_COUNTS,
)
scrambled = filter_rows(
    load_sweep(OUTPUT_ROOT, SCRAMBLED_SWEEP, "summary"),
    density=DENSITY, angle_k_ppp=STIFFNESS, E_att=E_VALUES, repeat_count=REPEAT_COUNTS,
)
periodic_reps = filter_rows(
    load_sweep(OUTPUT_ROOT, PERIODIC_SWEEP, "replicates"),
    density=DENSITY, angle_k_ppp=STIFFNESS, E_att=E_VALUES, repeat_count=REPEAT_COUNTS,
)
scrambled_reps = filter_rows(
    load_sweep(OUTPUT_ROOT, SCRAMBLED_SWEEP, "replicates"),
    density=DENSITY, angle_k_ppp=STIFFNESS, E_att=E_VALUES, repeat_count=REPEAT_COUNTS,
)
for data, label in ((periodic, "repeat"), (scrambled, "scrambled"),
                    (periodic_reps, "repeat"), (scrambled_reps, "scrambled")):
    data["sequence_class"] = label
summaries = pd.concat((periodic, scrambled), ignore_index=True)
independent = independent_units(
    pd.concat((periodic_reps, scrambled_reps), ignore_index=True)
)

## Cluster-size distribution by repeat length

In [ ]:
fig, axes = panels_by(
    periodic, by="E_att",
    plot_fn=partial(plot_cluster_bins, scheme="particle_percentage",
                    x="repeat_count", style=STYLE),
    style=STYLE, ncols=3, sharey=True, legend="bottom",
)
save_figure(fig, "length_01.png")
plt.show()

## Periodic — ranked clusters

In [ ]:
fig, axes = panels_by(
    periodic, by="E_att",
    plot_fn=partial(plot_ranked_clusters, x="repeat_count",
                    style=STYLE, annotate_molecules=True),
    style=STYLE, ncols=3, sharey=True, legend="bottom",
)
for ax in axes.flat:
    ax.set_ylim(0, 1)
save_figure(fig, "length_02.png")
plt.show()

## Scrambled — ranked clusters

In [ ]:
fig, axes = panels_by(
    scrambled, by="E_att",
    plot_fn=partial(plot_ranked_clusters, x="repeat_count",
                    style=STYLE, annotate_molecules=True),
    style=STYLE, ncols=3, sharey=True, legend="bottom",
)
for ax in axes.flat:
    ax.set_ylim(0, 1)
save_figure(fig, "length_03.png")
plt.show()

## Periodic and scrambled sequences — mean edge weight

In [ ]:
palette = {"repeat": "#2768EE", "scrambled": "#F12F48"}
fig, axes = panels_by(
    independent, by="E_att",
    plot_fn=partial(plot_metric_points, metric="mean_edge_weight",
                    x="repeat_count", hue="sequence_class",
                    palette=palette, style=STYLE),
    style=STYLE, ncols=3, width=4.1, height=3.8,
    sharey=True, legend="bottom",
)
save_figure(fig, "length_04.png")
plt.show()

## Permutation tests

In [ ]:
RUN_PERMUTATION_TESTS = True
if RUN_PERMUTATION_TESTS:
    from scipy.stats import permutation_test
    from statsmodels.stats.multitest import multipletests

    comparisons = []
    for (length, e_att), group in independent.groupby(["repeat_count", "E_att"]):
        a = group.loc[group.sequence_class == "repeat", "mean_edge_weight"].dropna().to_numpy()
        b = group.loc[group.sequence_class == "scrambled", "mean_edge_weight"].dropna().to_numpy()
        if len(a) >= 2 and len(b) >= 2:
            test = permutation_test(
                (a, b), statistic=lambda x, y: x.mean() - y.mean(),
                alternative="two-sided", permutation_type="independent",
                n_resamples=10_000, random_state=42,
            )
            comparisons.append({"repeat_count": length, "E_att": e_att, "p": test.pvalue})
    tests = pd.DataFrame(comparisons)
    if not tests.empty:
        tests["p_holm"] = multipletests(tests["p"], method="holm")[1]
        def format_p(row):
            p = row["p_holm"]
            stars = "**" if p < .01 else "*" if p < .05 else ""
            return f"{p:.4f}{stars}"

        tests["p_adj"] = tests.apply(format_p, axis=1)
        table = tests.pivot(index="repeat_count", columns="E_att", values="p_adj")
        table = table.reindex(index=REPEAT_COUNTS, columns=E_VALUES)
        table.index = [f"CAG{length}" for length in table.index]
        table.columns = [f"E_att = {strength:g}" for strength in table.columns]
        display(table.fillna("—"))

## Final-frame XY projections

The original GSD-based code below is optional (`SHOW_PROJECTIONS = False`). For the published notebook, embed the already-exported final-frame projection PNG here rather than requiring readers to download trajectories.


In [ ]:
SHOW_PROJECTIONS = False
if SHOW_PROJECTIONS:
    from src.plots.projections import plot_projection_sweep

    GSD_ROOT = ROOT.parent / "validation" / "bulk" / "output_dir"
    fig, axes = plot_projection_sweep(
        GSD_ROOT / PERIODIC_SWEEP, row_by="repeat_count",
        density=DENSITY, attractions=E_VALUES, style=STYLE,
    )
    figure_file = FIGURE_ROOT / "length_xy.png"
    figure_file.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(figure_file, dpi=250, bbox_inches="tight")
    print(f"Saved: {figure_file}")
    plt.show()